# End-to-End Pipeline: VLM Event Prediction, Moment Retrieval, and Video Summarisation

This notebook presents the end-to-end pipeline developed for the second assignment. The pipeline combines **Visual Language Model (VLM)-based event prediction**, **moment retrieval**, and **video summarisation** into one workflow for analysing video content.

The notebook follows the main stages of the pipeline:

1. **VLM event prediction**  
   The video is split into chunks and processed with a VLM to predict relevant events or actions from the visual content.

2. **Moment retrieval**  
   The predicted events are used as queries for pre-trained moment retrieval models, which identify the most relevant temporal segments in the video.

3. **Video summarisation**  
   Finally, the retrieved moments and event-level information are used to generate a concise summary of the video content.

The outputs shown throughout the notebook are included mainly for demonstration and readability. They should be interpreted as illustrative examples of the pipeline behavior, not as final experimental results or evaluation conclusions. The actual results are shown in the report and can be found in `outputs/intermediate/vlm_outputs`, `outputs/intermediate/retrieval_outputs`, and `outputs/final`.


# VLM Event Prediction

## 1. Setup 

In [1]:
from pathlib import Path


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir() and (candidate / "models").is_dir():
            return candidate

    return start


ROOT = find_project_root()
DATA_DIR = ROOT / "data"
VIDEO_DIR = DATA_DIR / "TVSum"
ANNOTATION_DIR = DATA_DIR / "annotations"
CACHE_DIR = ROOT / "cache"
OUTPUT_ROOT = ROOT / "outputs"
VLM_OUTPUT_DIR = OUTPUT_ROOT / "intermediate" / "vlm_outputs"
PREDICTION_DIR = VLM_OUTPUT_DIR / "vlm_final_prediction"
RETRIEVAL_OUTPUT_DIR = OUTPUT_ROOT / "intermediate" / "retrieval_outputs"
SUMMARY_OUTPUT_DIR = OUTPUT_ROOT / "final"
REQUIREMENTS_PATH = ROOT / "requirements.txt"

for directory in [CACHE_DIR, VLM_OUTPUT_DIR, PREDICTION_DIR, RETRIEVAL_OUTPUT_DIR, SUMMARY_OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")


Project root: C:\Users\Alex\Documents\Group2_Second_Assignment_CV2026\CompVision-SecondAssignment


In [3]:
!pip install -r "{REQUIREMENTS_PATH}"


Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu
  Cloning https://github.com/huggingface/transformers (to revision v4.49.0-SmolVLM-2) to C:\Users\Alex\AppData\Local\Temp\pip-req-build-raw0ytlq
  Resolved https://github.com/huggingface/transformers to commit 61e3ffd8148e68d879e3b2e1609fbb7d99621276
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Cloning https://github.com/line/lighthouse.git to C:\Users\Alex\AppData\Local\Temp\pip-req-build-ohfz294m
  Resolved https://github.com/line/lighthouse.git to commit d095eaa552cecef240897a8b750306b3b2a08740
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to 

  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers 'C:\Users\Alex\AppData\Local\Temp\pip-req-build-raw0ytlq'
  Running command git checkout -q 61e3ffd8148e68d879e3b2e1609fbb7d99621276
  Running command git clone --filter=blob:none --quiet https://github.com/line/lighthouse.git 'C:\Users\Alex\AppData\Local\Temp\pip-req-build-ohfz294m'
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git 'C:\Users\Alex\AppData\Local\Temp\pip-install-35ctjgh1\clip_50181621c15e42949cc9ebc77f74a8cb'


In [4]:
import torch
print(torch.cuda.is_available())
from transformers import AutoProcessor, AutoModelForImageTextToText
from moviepy.video.io.VideoFileClip import VideoFileClip
import os
import glob
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import math
import re
import num2words
import imageio_ffmpeg

os.environ["IMAGEIO_FFMPEG_EXE"] = imageio_ffmpeg.get_ffmpeg_exe()

False


The large model was run on Google Collab using a GPU and the results were saved. The smaller model can run on CPU, and hence was used for the end-to-end pipeline in this notebook. 

!To replicate the reports results, the large model needs to be used!

In [5]:
model_path = "HuggingFaceTB/SmolVLM2-256M-Video-Instruct"

#model_path = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"

processor = AutoProcessor.from_pretrained(model_path)

model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    torch_dtype=torch.float32,
    _attn_implementation="eager"
).to("cpu") # Changed from "cpu" to "cuda" to run on GPU

model.eval()

SmolVLMForConditionalGeneration(
  (model): SmolVLMModel(
    (vision_model): SmolVLMVisionTransformer(
      (embeddings): SmolVLMVisionEmbeddings(
        (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), padding=valid)
        (position_embedding): Embedding(1024, 768)
      )
      (encoder): SmolVLMEncoder(
        (layers): ModuleList(
          (0-11): 12 x SmolVLMEncoderLayer(
            (self_attn): SmolVLMVisionAttention(
              (k_proj): Linear(in_features=768, out_features=768, bias=True)
              (v_proj): Linear(in_features=768, out_features=768, bias=True)
              (q_proj): Linear(in_features=768, out_features=768, bias=True)
              (out_proj): Linear(in_features=768, out_features=768, bias=True)
            )
            (layer_norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (mlp): SmolVLMVisionMLP(
              (activation_fn): PytorchGELUTanh()
              (fc1): Linear(in_features=768, out_

In [6]:
# Our "golden" prompt for event extraction

PROMPT_EVENT_ONLY = """
Describe all the key events in the video.
Do not repeat events.
"""

In [7]:
# this function runs the model per input chunk
# need to define number of frames used per chunk + number of output tokens
def run_vlm_on_video(video_path, prompt, num_frames=8, max_new_tokens=500):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "video", "path": str(video_path)},
                {"type": "text", "text": prompt}
            ]
        }
    ]


    inputs = processor.apply_chat_template(
    messages,
    num_frames = num_frames,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    ).to(model.device) # Considering the size of the model, and the number of frames we are sampling are few. This is ultimately a choice that you must make


    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=max_new_tokens
        )

    generated_only = generated_ids[:, inputs["input_ids"].shape[1]:]

    answer = processor.batch_decode(
        generated_only,
        skip_special_tokens=True
    )[0]

    return answer.strip()


In [8]:
# spliting the entire video into equal length chunks (specified number)
def split_video_into_chunks(video_path, output_dir, num_chunks=10):

    os.makedirs(output_dir, exist_ok=True)

    # this finds if chunks already exist in the folder (for a given configuration)
    existing_chunks = sorted(glob.glob(os.path.join(output_dir, "chunk_*.mp4")))

    if existing_chunks:

        chunk_paths = []

        for chunk_path in existing_chunks:

            filename = os.path.basename(chunk_path)

            parts = (
                filename
                .replace("chunk_", "")
                .replace(".mp4", "")
                .split("_")
            )

            start = float(parts[0])
            end = float(parts[1])

            chunk_paths.append((chunk_path, start, end))

        print(f"Reusing {len(chunk_paths)} existing chunks from {output_dir}")

        return chunk_paths
    
    # if no chunks already exist, then split the video equally
    video = VideoFileClip(str(video_path))

    duration = video.duration
    chunk_duration = duration / num_chunks

    chunk_paths = []

    for i in range(num_chunks):
        

        # defining start and end times for a chunk
        start = i * chunk_duration

        if i == num_chunks - 1:
            end = duration
        else:
            end = (i + 1) * chunk_duration



        chunk_path = os.path.join(
            output_dir,
        f"chunk_{start:07.2f}_{end:07.2f}.mp4"
        )


        if hasattr(video, "subclipped"):
            clip = video.subclipped(start, end)
        else:
            clip = video.subclip(start, end)

        # creating the actual chunk
        clip.write_videofile(
            chunk_path,
            codec="libx264",
            audio=False,
            logger=None
        )

        clip.close()

        chunk_paths.append((chunk_path, start, end))

    video.close()

    return chunk_paths


##  2. Event extraction pipeline

In [9]:
# RUNNING MODEL WITH PROMPT ON CHUNKS AND SAVING OUTPUTS

# PARAMETERS
# num frames is the number of frames per chunk
# max tokens is the upper bound on number of tokens per generated output for each chunk

#CHOOSE VIDEO
video_number = 5
video_path = VIDEO_DIR / f"video_{video_number}.mp4"

video = VideoFileClip(str(video_path))
duration = video.duration

# parameters based on video number 
if video_number in [6,8]:

    num_chunks = 3

else:

    num_chunks = 1

if video_number in [6,8]:

    num_frames = 16
    max_new_tokens = 200

else:

    num_frames = 32
    max_new_tokens = 400

output_dir = CACHE_DIR / f"video_{video_number}_{num_chunks}_chunks"

chunks = split_video_into_chunks(
    video_path,
    output_dir=output_dir,
    num_chunks= num_chunks
)

all_outputs = []

for chunk_path, start, end in chunks:

    output = run_vlm_on_video(
        chunk_path,
        PROMPT_EVENT_ONLY,
        num_frames=num_frames,
        max_new_tokens=max_new_tokens 
    )

    all_outputs.append({
        "chunk": chunk_path,
        "start": start,
        "end": end,
        "vlm_output": output
    })

    print(f"\nCHUNK {start}-{end} seconds")
    print(output)

Reusing 1 existing chunks from C:\Users\Alex\Documents\Group2_Second_Assignment_CV2026\CompVision-SecondAssignment\cache\video_5_1_chunks

CHUNK 0.0-111.02 seconds
GMC Certified Service, replacing your tires.


In [11]:
# PARSES A GIVEN CHUNK 
def parse_vlm_events(text):

    parsed_events = []

    # remove quotes globally first
    text = text.replace('"', '').strip()  #   quotes were removed because VLM behaved bizzarly with them 
    

    # splitting when a symbol (. or ! or ?) is , indicating the start of a new sentence by the VLM 
    events = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text)

    
    for event in events:

        # removing unwanted characters (e.g. comma, dot etc.)
        cleaned_event = event.strip(" ,.-\n\t'")

        if cleaned_event:
            
            # saving parsed events to a list 
            parsed_events.append({
                "event": cleaned_event,
                "start_time": None,
                "end_time": None
            })

    return parsed_events

In [12]:
# going over all chunk outputs in chronological order
# and adding all predicted events per chunk together 
# to get predicted events per video
predicted_events = []

for item in all_outputs:
    parsed = parse_vlm_events(item["vlm_output"])
    predicted_events.extend([e["event"] for e in parsed])


In [13]:
print(predicted_events)

['GMC Certified Service, replacing your tires']


In [14]:
#Get our annotations
ANNOTATIONS_PATH = ANNOTATION_DIR / "2-TVSUM-video_5-6319373.txt"

video_annotations = []

#Read annotation file
with open(ANNOTATIONS_PATH, "r") as f:
    for line in f:
        #Split on comma and take the first element
        description = line.strip().split(",")[0]
        if description:

          video_annotations.append(description)
          print(description)

Branding start screen of GMC's tire replacement service
Person talking in front of a GMC car
Technician working on a car tire
Comparison of a tire with a good and bad tread
Displayed text recommending tire rotation every 7500 miles
Demonstration of checking tire tread depth using a tool and a coin
Tire spinning while mounted
People interacting near a car
Branding end screen of GMC's tire replacement service


In [15]:
print(len(video_annotations))

9


## 3. Evaluation using sentence transformers

In [16]:
# Embedding ground truth and predicted sentences to compute pairwise cosine similarity 

model_transformer = SentenceTransformer('all-MiniLM-L6-v2')


gt_embeddings = model_transformer.encode(video_annotations)
pred_embeddings = model_transformer.encode(predicted_events)


# computing pairwise cosine similarity between ground truth and predicted embeddings
sim_matrix = cosine_similarity(gt_embeddings, pred_embeddings) 

In [17]:
print(sim_matrix)

[[0.64300704]
 [0.31919417]
 [0.48553735]
 [0.36637396]
 [0.36390096]
 [0.3554467 ]
 [0.28407824]
 [0.12302731]
 [0.66367495]]


In [18]:
print(sim_matrix.shape)

(9, 1)


In [19]:
print(np.mean((sim_matrix)))

0.40047115


In [20]:
# Determining matches based on cosine similarity of the embeddings 

threshold = 0.5

matches = []
used_preds = set()

for i, gt in enumerate(video_annotations):

    best_j = np.argmax(sim_matrix[i])
    best_score = sim_matrix[i][best_j]

    # we can only match a ground truth event if a predicted one has a similarity higher than
    # threshold + it hasn't already been matched
    if best_score >= threshold and best_j not in used_preds:
        matches.append((i, gt, predicted_events[best_j], best_score))
        used_preds.add(best_j)

In [21]:
# NUMERICAL RESULTS OF COMPARING GROUND TRUTH AND PREDICTED EVENTS BASED ON MATCHES


true_positives = len(matches)
false_neg = len(video_annotations) - true_positives
false_pos = len(predicted_events) - true_positives

precision = true_positives / (true_positives + false_pos)
recall = true_positives / (true_positives + false_neg)
f1 = 2 * precision * recall / (precision + recall)


In [22]:
print(len(matches))

1


In [23]:
print(true_positives)
print(false_neg)
print(false_pos)

1
8
0


In [24]:
print(precision)
print(recall)
print(f1)

1.0
0.1111111111111111
0.19999999999999998


In [25]:
# FINDING THE MISSED EVENTS (i.e if for a ground truth event no predicted was found to be similar)

matched_gt_indices = set([m[0] for m in matches])

# an event is missed if it was not matched by cosine similarity 
missed_events = [
    (i, gt)
    for i, gt in enumerate(video_annotations)
    if i not in matched_gt_indices
]

print("\nMissed events:")

for i, e in missed_events:
    print(f"- Event {i}: {e}")


Missed events:
- Event 1: Person talking in front of a GMC car
- Event 2: Technician working on a car tire
- Event 3: Comparison of a tire with a good and bad tread
- Event 4: Displayed text recommending tire rotation every 7500 miles
- Event 5: Demonstration of checking tire tread depth using a tool and a coin
- Event 6: Tire spinning while mounted
- Event 7: People interacting near a car
- Event 8: Branding end screen of GMC's tire replacement service


In [26]:
# NON-MISSED EVENTS
print("\nNon-missed events:")

for i, gt, pred, score in matches:
    print(f"- Event {i}: {gt} (matched with '{pred}' with score: {score:.3f})")


Non-missed events:
- Event 0: Branding start screen of GMC's tire replacement service (matched with 'GMC Certified Service, replacing your tires' with score: 0.643)


In [27]:
# SAVING STEP 3 OUTPUTS TO A TEXT FILE 
text_file = []

text_file.append(f"Model: {model_path}")
text_file.append(f"Video: {video_number}")
text_file.append(f"Prompt given:\n{PROMPT_EVENT_ONLY}")
# text_file.append(f"Chunk Length: {chunk_length}")
text_file.append(f"Max Tokens used (per Chunk): {max_new_tokens}")
text_file.append(f"Number of Chunks: {num_chunks}")
text_file.append(f"Number of Frames (per Chunk): {num_frames}")
text_file.append(f"Threshold: {threshold}")

text_file.append("\n=== METRICS ===")
text_file.append(f"Precision: {precision:.3f}")
text_file.append(f"Recall:    {recall:.3f}")
text_file.append(f"F1-score:  {f1:.3f}")
text_file.append(f"Hallucinated events number: {false_pos}")
text_file.append(f"Correctly predicted events number: {true_positives}")
text_file.append(f"Missed out events number: {false_neg}")

text_file.append("\n=== PREDICTED EVENTS ===")

for i, event in enumerate(predicted_events):
    text_file.append(f"Predicted event {i}: {event}")


# MISSED EVENTS
matched_gt_indices = set([m[0] for m in matches])

missed_events = [
    (i, gt)
    for i, gt in enumerate(video_annotations)
    if i not in matched_gt_indices
]

text_file.append("\n=== MISSED EVENTS ===")

for i, e in missed_events:
    text_file.append(f"Missed event {i}: {e}")


# NON-MISSED EVENTS
text_file.append("\n=== MATCHED EVENTS ===")

for i, gt, pred, score in matches:
    text_file.append(
        f"Matched GT event {i}: {gt}"
    )
    text_file.append(
        f"    Predicted: {pred}"
    )
    text_file.append(
        f"    Similarity score: {score:.3f}"
    )


# SAVE FILE
VLM_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_path = VLM_OUTPUT_DIR / f"{video_number}_results_step3.txt"

with open(output_path, "a") as f:
    f.write("\n".join(text_file))

print(f"Results saved to: {output_path}")

Results saved to: C:\Users\Alex\Documents\Group2_Second_Assignment_CV2026\CompVision-SecondAssignment\outputs\intermediate\vlm_outputs\5_results_step3.txt


---

# Moment Retrieval Using Pre-trained Models

- Video 5 and 7 are approximately 120 seconds, so they can be encoded directly by Lighthouse.
- Video 6 and 8 are about 300 seconds, while Lighthouse's benchmark-style raw-video path is safest below about 150 seconds, so the notebook chunks video 6 and 8.

!In this notebook, only video 5 was run, with the same prediction used in the report!

## 1. Setup

In [28]:
from pathlib import Path
import json
import shutil
import sys
import warnings
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "")
os.environ.setdefault("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", "1")
import cv2
import ffmpeg
import ffmpeg.nodes as ffmpeg_nodes
import pandas as pd
import torch.nn.functional as F
from IPython.display import Video, display
try:
    from moviepy import VideoFileClip
except ImportError:
    from moviepy.editor import VideoFileClip

FFMPEG_EXE = imageio_ffmpeg.get_ffmpeg_exe()
if not hasattr(ffmpeg, "_codex_original_run"):
    ffmpeg._codex_original_run = ffmpeg.run
    ffmpeg._codex_original_run_async = ffmpeg.run_async

def _run_with_imageio_ffmpeg(stream_spec, cmd=FFMPEG_EXE, **kwargs):
    return ffmpeg._codex_original_run(stream_spec, cmd=cmd, **kwargs)

def _run_async_with_imageio_ffmpeg(stream_spec, cmd=FFMPEG_EXE, **kwargs):
    return ffmpeg._codex_original_run_async(stream_spec, cmd=cmd, **kwargs)

ffmpeg.run = _run_with_imageio_ffmpeg
ffmpeg.run_async = _run_async_with_imageio_ffmpeg
ffmpeg_nodes.OutputStream.run = _run_with_imageio_ffmpeg
ffmpeg_nodes.OutputStream.run_async = _run_async_with_imageio_ffmpeg

from lighthouse.frame_loaders.base_loader import BaseLoader

def _video_info_with_opencv(self, video_path):
    capture = cv2.VideoCapture(str(video_path))
    try:
        if not capture.isOpened():
            return None

        width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps_value = float(capture.get(cv2.CAP_PROP_FPS) or 0.0)
        frames_length = int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)

        if width <= 0 or height <= 0 or fps_value <= 0 or frames_length <= 0:
            return None

        return {
            "duration": frames_length / fps_value,
            "frames_length": frames_length,
            "fps": max(1, math.floor(fps_value)),
            "height": height,
            "width": width,
        }
    finally:
        capture.release()

BaseLoader._video_info = _video_info_with_opencv

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="CUDA initialization:.*")

#display all rows and columns in pandas DataFrames
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

if "find_project_root" not in globals():
    def find_project_root(start=None):
        start = Path(start or Path.cwd()).resolve()

        for candidate in [start, *start.parents]:
            if (candidate / "data").is_dir() and (candidate / "models").is_dir():
                return candidate

        return start

ROOT = find_project_root()
DATA_DIR = ROOT / "data"
VIDEO_DIR = DATA_DIR / "TVSum"
ANNOTATION_DIR = DATA_DIR / "annotations"
CACHE_DIR = ROOT / "cache"
MODEL_SOURCE = ROOT / "models"
OUTPUT_ROOT = ROOT / "outputs"
VLM_OUTPUT_DIR = OUTPUT_ROOT / "intermediate" / "vlm_outputs"
PREDICTION_DIR = VLM_OUTPUT_DIR / "vlm_final_prediction"
PREDICTION_SOURCE = PREDICTION_DIR
RETRIEVAL_OUTPUT_DIR = OUTPUT_ROOT / "intermediate" / "retrieval_outputs"
SUMMARY_OUTPUT_DIR = OUTPUT_ROOT / "final"
OUTPUT_DIR = RETRIEVAL_OUTPUT_DIR

for directory in [CACHE_DIR, VLM_OUTPUT_DIR, PREDICTION_DIR, RETRIEVAL_OUTPUT_DIR, SUMMARY_OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

VIDEO_IDS_TO_RUN = [5]
DEVICE = "cpu"

In [29]:
from lighthouse.models import CGDETRPredictor, QDDETRPredictor, MomentDETRPredictor


MODEL_CONFIGS = {
    "CG-DETR": {
        "predictor_cls": CGDETRPredictor,
        "checkpoint": MODEL_SOURCE/ "clip_cg_detr_qvhighlight.ckpt",
        "feature_name": "clip",
        "slowfast_path": None,
        "selected": True,
    },
    "QD-DETR": {
        "predictor_cls": QDDETRPredictor,
        "checkpoint": MODEL_SOURCE / "qd_detr_qvhighlight_videoonly.ckpt",
        # This checkpoint expects 2818-dim video input:
        # 512 CLIP + 2304 SlowFast + 2 temporal endpoint features.
        "feature_name": "clip_slowfast",
        "slowfast_path": MODEL_SOURCE / "SLOWFAST_8x8_R50.pkl",
        "selected": True,
    },
    "Moment-DETR": {
        "predictor_cls": MomentDETRPredictor,
        "checkpoint": MODEL_SOURCE/ "moment_detr_qvhighlight_clip.ckpt",
        "feature_name": "clip",
        "slowfast_path": None,
        "selected": True,
    },
}

## 2. Parse Videos and Annotations

The annotation files are parsed into event text, start/end seconds, and subjectivity score. The default query table keeps events with score `>= 1`, because those are the objectively important/essential moments most relevant to a summary.

In [30]:
def mmss_to_seconds(value):

    value = value.strip()

    minutes, seconds = value.split(":")

    return int(minutes) * 60 + int(seconds)


def seconds_to_mmss(seconds):

    seconds = max(0, int(round(seconds)))

    return f"{seconds // 60:02d}:{seconds % 60:02d}"

TIME_RE = re.compile(r"(\d{1,2}:\d{2})\s*(?:-|\u2013|\u2014|\?|\uFFFD)+\s*(\d{1,2}:\d{2})")

def infer_video_id(path):

    name = Path(path).name.lower()

    video_match = re.search(r"video[_-]?(\d+)", name)

    if video_match:

        return int(video_match.group(1))

    leading_number_match = re.search(r"^(\d+)(?:\D|$)", name)

    if leading_number_match:

        return int(leading_number_match.group(1))

    return None


def should_keep_video_id(video_id):

    if video_id is None:

        return False

    if "VIDEO_IDS_TO_RUN" not in globals():

        return True

    return int(video_id) in set(int(video_id) for video_id in VIDEO_IDS_TO_RUN)


def parse_annotation_line(line, source_file, row_index):

    cleaned = line.strip().replace("\u2013", "-").replace("\u2014", "-").replace("\ufffd", "-")

    if not cleaned:

        return None

    time_match = TIME_RE.search(cleaned)

    score_match = re.search(r",\s*(-?\d+)\s*$", cleaned)

    if not time_match or not score_match:

        return None

    event = cleaned[: time_match.start()].strip(" ,")

    start_s = mmss_to_seconds(time_match.group(1))

    end_s = mmss_to_seconds(time_match.group(2))

    score = int(score_match.group(1))

    video_id = infer_video_id(source_file)


    return {

        "video_id": video_id,
        "source_file": source_file.name,
        "row_index": row_index,
        "event": event,
        "gt_start": start_s,
        "gt_end": end_s,
        "gt_time": f"{seconds_to_mmss(start_s)} - {seconds_to_mmss(end_s)}",
        "subjectivity": score,

    }


def parse_prediction_line(line, source_file, row_index):

    cleaned = line.strip().replace("\u2013", "-").replace("\u2014", "-").replace("\ufffd", "-")

    if not cleaned:

        return None

    time_match = TIME_RE.search(cleaned)

    if not time_match:

        return None

    event = cleaned[: time_match.start()].strip(" ,")

    event = re.sub(r"^Predicted event\s+\d+\s*:\s*", "", event, flags=re.IGNORECASE).strip()

    start_s = mmss_to_seconds(time_match.group(1))

    end_s = mmss_to_seconds(time_match.group(2))

    subjectivity = 1

    suffix = cleaned[time_match.end():].strip(" ,")

    if suffix:

        score_match = re.search(r"(-?\d+)\s*$", suffix)

        if score_match:

            subjectivity = int(score_match.group(1))

    video_id = infer_video_id(source_file)

    return {

        "video_id": video_id,
        "source_file": source_file.name,
        "row_index": row_index,
        "event": event,
        "gt_start": start_s,
        "gt_end": end_s,
        "gt_time": f"{seconds_to_mmss(start_s)} - {seconds_to_mmss(end_s)}",
        "subjectivity": subjectivity,

    }


def iter_text_files(path_or_dir):

    path_or_dir = Path(path_or_dir)

    if path_or_dir.is_file():

        return [path_or_dir]

    if path_or_dir.is_dir():

        return sorted(path_or_dir.glob("*.txt"))

    return []


def load_predictions(prediction_source):

    rows = []
    skipped_lines = []

    for path in iter_text_files(prediction_source):

        video_id = infer_video_id(path)

        if not should_keep_video_id(video_id):

            continue

        for row_index, line in enumerate(path.read_text(encoding="utf-8", errors="replace").splitlines(), 1):

            parsed = parse_prediction_line(line, path, row_index)

            if parsed:

                rows.append(parsed)
            elif line.strip():

                skipped_lines.append(f"{path.name}:{row_index}")

    if skipped_lines:

        preview = ", ".join(skipped_lines[:5])
        suffix = "..." if len(skipped_lines) > 5 else ""
        print(
            f"Skipped {len(skipped_lines)} prediction line(s) without a valid time range: "
            f"{preview}{suffix}. Expected format: event text, MM:SS - MM:SS, subjectivity"
        )

    return pd.DataFrame(rows, columns=["video_id", "source_file", "row_index", "event", "gt_start", "gt_end", "gt_time", "subjectivity"])


def load_annotations(annotation_dir):

    rows = []

    for path in sorted(annotation_dir.glob("*.txt")):

        video_id = infer_video_id(path)

        if not should_keep_video_id(video_id):

            continue

        for row_index, line in enumerate(path.read_text(encoding="utf-8", errors="replace").splitlines(), 1):

            parsed = parse_annotation_line(line, path, row_index)

            if parsed:

                rows.append(parsed)

    return pd.DataFrame(rows)


annotations_df = load_annotations(ANNOTATION_DIR)
predictions_df = load_predictions(PREDICTION_SOURCE)

predictions_df 

,video_id,source_file,row_index,event,gt_start,gt_end,gt_time,subjectivity
0,5,video5_step3_predictions.txt,1,The video begins with a title screen displayin...,0,4,00:00 - 00:04,1
1,5,video5_step3_predictions.txt,2,"The man, identified as Dave Campbell, provides...",6,18,00:06 - 00:18,1
2,5,video5_step3_predictions.txt,3,The video continues with a close-up of a tire ...,21,37,00:21 - 00:37,1
3,5,video5_step3_predictions.txt,4,The video emphasizes the importance of rotatin...,38,43,00:38 - 00:43,1
4,5,video5_step3_predictions.txt,5,The final segment shows a man in a white shirt...,92,97,01:32 - 01:37,1
5,5,video5_step3_predictions.txt,6,The video concludes with a title screen displa...,107,110,01:47 - 01:50,1


In [31]:
# Use Step 3 predicted events when available; otherwise fall back to manual annotations.
query_source_df = predictions_df if not predictions_df.empty else annotations_df
query_source_name = "Step 3 predictions" if not predictions_df.empty else "manual annotations"

query_table = (
    query_source_df[
        (query_source_df["video_id"].isin(VIDEO_IDS_TO_RUN))
        & (query_source_df["subjectivity"] >= 1)
    ]
    .sort_values(["video_id", "gt_start", "gt_end", "source_file"])
    .reset_index(drop=True)
)

print(f"Using {query_source_name}")
print(query_table.groupby("video_id").size())
query_table[["video_id", "event", "gt_time", "subjectivity", "source_file"]]

Using Step 3 predictions
video_id
5    6
dtype: int64


,video_id,event,gt_time,subjectivity,source_file
0,5,The video begins with a title screen displayin...,00:00 - 00:04,1,video5_step3_predictions.txt
1,5,"The man, identified as Dave Campbell, provides...",00:06 - 00:18,1,video5_step3_predictions.txt
2,5,The video continues with a close-up of a tire ...,00:21 - 00:37,1,video5_step3_predictions.txt
3,5,The video emphasizes the importance of rotatin...,00:38 - 00:43,1,video5_step3_predictions.txt
4,5,The final segment shows a man in a white shirt...,01:32 - 01:37,1,video5_step3_predictions.txt
5,5,The video concludes with a title screen displa...,01:47 - 01:50,1,video5_step3_predictions.txt


## 3. Retrieval Utilities

In [32]:
MAX_DIRECT_VIDEO_SECONDS = 150
CHUNK_SECONDS = 30
TOP_K_PER_QUERY = 5
NMS_IOU_THRESHOLD = 0.70

QUERY_VARIANT_STRATEGIES_TO_RUN = [
    "raw_vlm",
    "cleaned",
    "short_action",
    "the_video_shows",
    "find_moment",
    "screen_text",
]

QUERY_VARIANT_DESCRIPTIONS = {
    "raw_vlm": "Original Step 3 event sentence.",
    "cleaned": "Remove VLM narration such as 'the video begins with'.",
    "short_action": "Keep the most direct subject/action/object clause.",
    "the_video_shows": "Wrap the cleaned event as 'The video shows ...'.",
    "find_moment": "Wrap the cleaned event as 'Find the moment showing ...'.",
    "screen_text": "Emphasize quoted text or on-screen text when present.",
}


def temporal_iou(a_start, a_end, b_start, b_end):

    inter = max(0.0, min(a_end, b_end) - max(a_start, b_start))

    union = max(a_end, b_end) - min(a_start, b_start)
    
    return inter / union if union > 0 else 0.0


def nms_windows(windows, iou_threshold=0.70, max_keep=5):

    ordered = sorted(windows, key=lambda item: item["score"], reverse=True)

    kept = []

    for item in ordered:

        overlaps = [
            temporal_iou(item["pred_start"], item["pred_end"], prev["pred_start"], prev["pred_end"])
            for prev in kept
        ]

        if not overlaps or max(overlaps) < iou_threshold:

            kept.append(item)

        if len(kept) >= max_keep:

            break

    return kept


def list_existing_chunks(chunk_dir):

    chunk_dir = Path(chunk_dir)

    if not chunk_dir.exists():

        return []
    
    chunks = []

    for path in sorted(chunk_dir.glob("chunk_*.mp4")):

        match = re.match(r"chunk_(\d+(?:\.\d+)?)_(\d+(?:\.\d+)?)\.mp4$", path.name)

        if not match:

            continue

        start = float(match.group(1))
        end = float(match.group(2))

        if end <= start:

            continue

        chunks.append({"path": path, "offset": start, "end": end})

    return sorted(chunks, key=lambda item: (item["offset"], item["end"]))


def chunks_cover_video(chunks, video_duration, max_gap_seconds=1.5):

    if not chunks or video_duration <= 0:

        return False

    cursor = 0.0

    for chunk in sorted(chunks, key=lambda item: item["offset"]):

        start = max(0.0, min(float(video_duration), float(chunk["offset"])))
        end = max(0.0, min(float(video_duration), float(chunk["end"])))

        if end <= start:

            continue

        if start > cursor + max_gap_seconds:

            return False

        cursor = max(cursor, end)

    return cursor >= float(video_duration) - max_gap_seconds


def find_existing_chunk_sets(video_id, video_duration):

    candidate_dirs = [CACHE_DIR / f"video_{video_id}_chunks"]

    for pattern in [
        f"video_{video_id}_chunks*",
        f"video_{video_id}_*_chunks",
        f"video_{video_id}_*_chunks*",
    ]:

        candidate_dirs.extend(sorted(path for path in CACHE_DIR.glob(pattern) if path.is_dir()))

    seen = set()
    chunk_sets = []

    for chunk_dir in candidate_dirs:

        chunk_dir = Path(chunk_dir)
        resolved = chunk_dir.resolve()

        if resolved in seen:

            continue

        seen.add(resolved)

        chunks = list_existing_chunks(chunk_dir)

        if not chunks:

            continue

        chunk_sets.append(
            {
                "chunk_dir": chunk_dir,
                "chunks": chunks,
                "complete": chunks_cover_video(chunks, video_duration),
                "last_end": max(chunk["end"] for chunk in chunks),
            }
        )

    return chunk_sets


def choose_existing_chunks(video_id, video_duration):

    chunk_sets = find_existing_chunk_sets(video_id, video_duration)

    complete_sets = [chunk_set for chunk_set in chunk_sets if chunk_set["complete"]]

    if not complete_sets:

        return []

    # Prefer the most granular complete chunk set, since retrieval scores are local to each segment.
    selected = max(complete_sets, key=lambda item: (len(item["chunks"]), item["last_end"]))
    print(f"Using existing chunks from {selected['chunk_dir'].relative_to(ROOT)}")

    return selected["chunks"]


def get_video_duration(video_path) :

    clip = VideoFileClip(str(video_path))

    try:

        return float(clip.duration)
    
    finally:

        clip.close()


def get_video_segments(video_id: int):

    video_path = VIDEO_DIR / f"video_{video_id}.mp4"

    duration = get_video_duration(video_path)

    existing_chunks = choose_existing_chunks(video_id, duration)

    if existing_chunks:

        return existing_chunks, duration

    if duration <= MAX_DIRECT_VIDEO_SECONDS:

        return [{"path": video_path, "offset": 0.0, "end": duration}], duration
    

    chunk_dir = CACHE_DIR / f"video_{video_id}_chunks"

    chunks = list_existing_chunks(chunk_dir)

    if not chunks:

        chunks = split_video_into_chunks(video_path, chunk_dir, CHUNK_SECONDS)

    return chunks, duration

def normalize_query_text(text):

    return re.sub(r"\s+", " ", str(text)).strip(" .,")


def lower_first(text):

    text = normalize_query_text(text)

    return text[:1].lower() + text[1:] if text else text


def clean_vlm_event(event):

    text = normalize_query_text(event)

    prefix_re = re.compile(
        r"^(the video begins with|the video starts with|the video continues with|"
        r"the final segment shows|the video concludes with|it transitions to|"
        r"the segment shows|this segment shows)\s+",
        flags=re.IGNORECASE,
    )

    previous = None

    while previous != text:

        previous = text

        text = prefix_re.sub("", text).strip(" .,;")

    return text or normalize_query_text(event)


def short_action_query(event):

    text = clean_vlm_event(event)

    # Step 3 descriptions often contain multiple clauses; the first visual clause is usually the cleanest query.
    text = re.split(r"\s*(?:, followed by|, with|\. It |;|,)\s*", text, maxsplit=1)[0]

    return normalize_query_text(text)


def screen_text_query(event):

    text = clean_vlm_event(event)

    quoted = re.findall(r'"([^"]+)"', text)

    if quoted:

        return normalize_query_text("screen text " + " ".join(quoted))

    if re.search(r"\b(text|title|screen|website|logo|display|appears?)\b", text, flags=re.IGNORECASE):

        return normalize_query_text(text)

    return short_action_query(text)


def build_query_variant(event, strategy):

    raw = normalize_query_text(event)

    cleaned = clean_vlm_event(raw)

    builders = {
        "raw_vlm": lambda: raw,
        "cleaned": lambda: cleaned,
        "short_action": lambda: short_action_query(cleaned),
        "the_video_shows": lambda: f"The video shows {lower_first(cleaned)}",
        "find_moment": lambda: f"Find the moment showing {lower_first(cleaned)}",
        "screen_text": lambda: screen_text_query(cleaned),
    }

    if strategy not in builders:

        raise ValueError(f"Unknown query variant strategy: {strategy}")

    return normalize_query_text(builders[strategy]())


def make_query_variants(event, strategy="raw_vlm"):

    if strategy == "all_unique":

        variants = [build_query_variant(event, name) for name in QUERY_VARIANT_DESCRIPTIONS]

    else:

        variants = [build_query_variant(event, strategy)]

    return list(dict.fromkeys([variant for variant in variants if variant]))


def preview_query_variants(query_table, strategies=QUERY_VARIANT_STRATEGIES_TO_RUN, n=12):

    rows = []

    for query_row in query_table.head(n).itertuples(index=False):

        for strategy in strategies:

            for variant in make_query_variants(query_row.event, strategy):

                rows.append(
                    {
                        "video_id": query_row.video_id,
                        "row_index": query_row.row_index,
                        "gt_time": query_row.gt_time,
                        "query_variant_strategy": strategy,
                        "query_variant": variant,
                    }
                )

    return pd.DataFrame(rows)


preview_query_variants(query_table, n=3)

,video_id,row_index,gt_time,query_variant_strategy,query_variant
0,5,1,00:00 - 00:04,raw_vlm,The video begins with a title screen displayin...
1,5,1,00:00 - 00:04,cleaned,"a title screen displaying ""GMC Certified Servi..."
2,5,1,00:00 - 00:04,short_action,"a title screen displaying ""GMC Certified Servi..."
3,5,1,00:00 - 00:04,the_video_shows,"The video shows a title screen displaying ""GMC..."
4,5,1,00:00 - 00:04,find_moment,Find the moment showing a title screen display...
5,5,1,00:00 - 00:04,screen_text,screen text GMC Certified Service Replacing Yo...
6,5,2,00:06 - 00:18,raw_vlm,"The man, identified as Dave Campbell, provides..."
7,5,2,00:06 - 00:18,cleaned,"The man, identified as Dave Campbell, provides..."
8,5,2,00:06 - 00:18,short_action,The man
9,5,2,00:06 - 00:18,the_video_shows,"The video shows the man, identified as Dave Ca..."


In [33]:
def load_available_models(model_configs):

    models = {}
    skipped = []

    # Load preferred models first so the output order is predictable.
    preferred_order = ["CG-DETR", "Moment-DETR"]
    ordered_names = preferred_order + [name for name in model_configs if name not in preferred_order]

    for name in ordered_names:

        cfg = model_configs[name]

        if not cfg["selected"]:
            continue

        checkpoint = cfg["checkpoint"]

        # Skip missing checkpoints instead of stopping the whole pipeline.
        if not checkpoint.exists():

            skipped.append(
                {
                    "model": name,
                    "reason": f"Missing checkpoint: {checkpoint.relative_to(ROOT)}",
                }
            )
            continue

        slowfast_path = cfg.get("slowfast_path")

        # SlowFast feature models need the extra backbone checkpoint.
        if cfg["feature_name"] in {"clip_slowfast", "clip_slowfast_pann"}:

            if slowfast_path is None or not slowfast_path.exists():

                skipped.append(
                    {
                        "model": name,
                        "reason": "Missing SlowFast backbone: SLOWFAST_8x8_R50.pkl",
                    }
                )
                continue

        print(f"Loading {name} from {checkpoint.name} ...")

        models[name] = cfg["predictor_cls"](
            str(checkpoint),
            device=DEVICE,
            feature_name=cfg["feature_name"],
            slowfast_path=None if slowfast_path is None else str(slowfast_path),
        )

    return models, pd.DataFrame(skipped)


models, skipped_models = load_available_models(MODEL_CONFIGS)
print("Loaded:", list(models))
skipped_models

Loading CG-DETR from clip_cg_detr_qvhighlight.ckpt ...
Loading Moment-DETR from moment_detr_qvhighlight_clip.ckpt ...
Loading QD-DETR from qd_detr_qvhighlight_videoonly.ckpt ...
Loaded: ['CG-DETR', 'Moment-DETR', 'QD-DETR']


""


In [12]:
RETRIEVAL_RESULT_COLUMNS = [
    "model",
    "event",
    "query_variant_strategy",
    "query_variant",
    "gt_time",
    "subjectivity",
    "video_id",
    "source_file",
    "row_index",
    "gt_start",
    "gt_end",
    "segment_path",
    "segment_offset",
    "rank_in_segment",
    "pred_time",
    "pred_start",
    "pred_end",
    "score",
    "global_rank",
    "iou",
]


def _prediction_windows(prediction):
    if prediction is None:
        return None

    if isinstance(prediction, dict):
        return prediction.get("pred_relevant_windows")

    windows = getattr(prediction, "pred_relevant_windows", None)
    if windows is not None:
        return windows

    return prediction


def retrieve_for_model(model_name, model, query_table, query_variant_strategy="raw_vlm", encoded_cache=None):

    rows = []
    skipped_predictions = []

    # Cache encoded video features so each video/chunk is encoded only once per model.
    if encoded_cache is None:
        encoded_cache = {}

    # Process queries video by video.
    for video_id, video_queries in query_table.groupby("video_id"):

        # Get either the full video or its chunks, plus the full video duration.
        segments, video_duration = get_video_segments(video_id)

        print(f"{model_name} | video {video_id}: {len(video_queries)} queries, {len(segments)} segment(s)")

        # Loop over each full video segment or video chunk.
        for segment in segments:

            segment_path = segment["path"]

            # Create a unique cache key for this video/chunk.
            cache_key = str(segment_path.resolve())

            # If this segment was not encoded yet, encode it now.
            if cache_key not in encoded_cache:

                # Print which segment is being encoded.
                print(f"  encoding {segment_path.relative_to(ROOT)}")

                # Extract video features using the current model's encoder.
                encoded_cache[cache_key] = model.encode_video(str(segment_path))

            # Reuse the encoded video features.
            encoded_video = encoded_cache[cache_key]

            # Loop over each event/query for this video.
            for query_row in video_queries.itertuples(index=False):

                # Store predictions for this event/query.
                all_windows = []

                # Use one or more phrasings of the event query.
                for variant in make_query_variants(query_row.event, query_variant_strategy):

                    prediction = model.predict(variant, encoded_video)
                    windows = _prediction_windows(prediction)

                    if windows is None:
                        skipped_predictions.append(
                            {
                                "model": model_name,
                                "video_id": video_id,
                                "segment_path": str(segment_path.relative_to(ROOT)),
                                "event": query_row.event,
                                "query_variant_strategy": query_variant_strategy,
                                "query_variant": variant,
                                "reason": "predict returned no windows",
                            }
                        )
                        continue

                    # Loop over the model's predicted windows.
                    for rank, (start, end, score) in enumerate(windows, 1):

                        # Convert local segment start time into full-video start time.
                        pred_start = max(0.0, min(video_duration, float(start) + segment["offset"]))

                        # Convert local segment end time into full-video end time.
                        pred_end = max(0.0, min(video_duration, float(end) + segment["offset"]))

                        # Skip invalid predictions where the end comes before the start.
                        if pred_end <= pred_start:
                            continue

                        all_windows.append(
                            {
                              
                                "model": model_name,
                                "video_id": video_id,
                                "source_file": query_row.source_file,
                                "row_index": query_row.row_index,
                                "event": query_row.event,
                                "query_variant_strategy": query_variant_strategy,
                                "query_variant": variant,
                                "gt_start": float(query_row.gt_start),
                                "gt_end": float(query_row.gt_end),
                                "gt_time": query_row.gt_time,
                                "subjectivity": int(query_row.subjectivity),
                                "segment_path": str(segment_path.relative_to(ROOT)),
                                "segment_offset": segment["offset"],
                                "rank_in_segment": rank,
                                "pred_start": pred_start,
                                "pred_end": pred_end,
                                "pred_time": f"{seconds_to_mmss(pred_start)} - {seconds_to_mmss(pred_end)}",
                                "score": float(score),
                            }
                        )

                # Add this query's predictions to the full result list.
                rows.extend(all_windows)

    raw = pd.DataFrame(rows)

    if raw.empty:
        return pd.DataFrame(columns=RETRIEVAL_RESULT_COLUMNS)

    final_rows = []

    group_cols = ["model", "query_variant_strategy", "video_id", "source_file", "row_index", "event"]

    # Group all predictions belonging to the same query.
    for _, group in raw.groupby(group_cols, sort=False):

        # Sort predictions by confidence score, highest first.
        windows = group.sort_values("score", ascending=False).to_dict("records")

        # Remove highly overlapping duplicate windows and keep top K.
        kept = nms_windows(windows, iou_threshold=NMS_IOU_THRESHOLD, max_keep=TOP_K_PER_QUERY)

        # Give the remaining predictions a global rank.
        for global_rank, item in enumerate(kept, 1):

            # Store rank after NMS.
            item["global_rank"] = global_rank

            # Compute overlap between prediction and ground truth.
            item["iou"] = temporal_iou(
                item["pred_start"],
                item["pred_end"],
                item["gt_start"],
                item["gt_end"],
            )

            final_rows.append(item)

    # Return final predictions as a dataframe.
    return pd.DataFrame(final_rows, columns=RETRIEVAL_RESULT_COLUMNS)


## 4. Run Retrieval

This is the main Step 4 execution cell. On CPU, the first run is slower because video features are encoded. Subsequent query predictions reuse cached features inside the cell.

In [ ]:
all_results = []

for model_name, model in models.items():

    encoded_cache = {}

    for query_variant_strategy in QUERY_VARIANT_STRATEGIES_TO_RUN:

        print(f"Running {model_name} with query strategy: {query_variant_strategy}")

        model_results = retrieve_for_model(
            model_name,
            model,
            query_table,
            query_variant_strategy=query_variant_strategy,
            encoded_cache=encoded_cache,
        )

        if not model_results.empty:

            all_results.append(model_results)

if all_results:
    results_df = pd.concat(all_results, ignore_index=True)
    retrieval_predictions_path = RETRIEVAL_OUTPUT_DIR / "step4_retrieval_predictions.csv"
    results_df.to_csv(retrieval_predictions_path, index=False)
    print(f"Saved retrieval predictions to {retrieval_predictions_path.relative_to(ROOT)}")
else:
    results_df = pd.DataFrame()
    print("No retrieval predictions to save.")

print(f"Prediction rows: {len(results_df)}")
results_df.head()

In [ ]:
if "query_variant_strategy" not in results_df.columns:
    results_df["query_variant_strategy"] = "raw_vlm"

ranked_results = results_df.sort_values(
    ["model", "query_variant_strategy", "video_id", "global_rank"]
)

best_by_query = (
    ranked_results.groupby(["model", "query_variant_strategy", "video_id", "event"], as_index=False)
    .agg(
        gt_time=("gt_time", "first"),
        subjectivity=("subjectivity", "first"),
        top1_query_variant=("query_variant", "first"),
        top1_time=("pred_time", "first"),
        top1_score=("score", "first"),
        top1_iou=("iou", "first"),
        best_iou_at_k=("iou", "max"),
    )
)

summary_df = (
    best_by_query.groupby(["model", "query_variant_strategy", "video_id"], as_index=False)
    .agg(
        queries=("event", "count"),
        mean_top1_iou=("top1_iou", "mean"),
        mean_best_iou_at_k=("best_iou_at_k", "mean"),
        recall_top1_iou_030=("top1_iou", lambda values: (values >= 0.30).mean()),
        recall_at_k_iou_030=("best_iou_at_k", lambda values: (values >= 0.30).mean()),
        recall_at_k_iou_050=("best_iou_at_k", lambda values: (values >= 0.50).mean()),
    )
    .sort_values(["model", "video_id", "mean_top1_iou", "mean_best_iou_at_k"], ascending=[True, True, False, False])
)

best_strategy_df = (
    summary_df.sort_values(["model", "video_id", "mean_top1_iou", "mean_best_iou_at_k"], ascending=[True, True, False, False])
    .groupby(["model", "video_id"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

display(best_strategy_df)
display(summary_df)
display(best_by_query.sort_values(["model", "query_variant_strategy", "video_id", "best_iou_at_k"], ascending=[True, True, True, False]).head(12))

video_ids = [int(video_id) for video_id in VIDEO_IDS_TO_RUN]
output_suffix = "" if video_ids == [5] else (str(video_ids[0]) if len(video_ids) == 1 else "_all")
best_by_query_path = RETRIEVAL_OUTPUT_DIR / f"step4_best_by_query{output_suffix}.csv"
summary_metrics_path = RETRIEVAL_OUTPUT_DIR / f"step4_summary_metrics{output_suffix}.csv"
best_strategy_path = RETRIEVAL_OUTPUT_DIR / f"step4_best_query_strategy{output_suffix}.csv"
best_by_query.to_csv(best_by_query_path, index=False)
summary_df.to_csv(summary_metrics_path, index=False)
best_strategy_df.to_csv(best_strategy_path, index=False)
print(f"Saved per-query results to {best_by_query_path.relative_to(ROOT)}")
print(f"Saved summary metrics to {summary_metrics_path.relative_to(ROOT)}")
print(f"Saved best query strategy summary to {best_strategy_path.relative_to(ROOT)}")

## 6. Video Summary

This final step turns the retrieved moments into a compact video-level summary. It uses the best available retrieval model, keeps one high-scoring window per salient event, removes overlapping duplicate windows, orders the selected moments chronologically, and exports both a readable text summary and optional concatenated summary clips.

In [ ]:
SUMMARY_PREFERRED_MODEL = "CG-DETR" 
SUMMARY_PREFERRED_QUERY_STRATEGY = "screen_text" 
SUMMARY_MOMENTS_PER_VIDEO = 6
SUMMARY_PADDING_SECONDS = 1.0
SUMMARY_OVERLAP_THRESHOLD = 0.5


def pick_summary_model(results, preferred_model=SUMMARY_PREFERRED_MODEL):
    available = sorted(results["model"].dropna().unique())

    if preferred_model in available:
        return preferred_model

    if "iou" in results.columns and not results["iou"].dropna().empty:
        return results.groupby("model")["iou"].mean().sort_values(ascending=False).index[0]

    return available[0] if available else None


def pick_summary_query_strategy(results, model_name, video_id=None, preferred_strategy=SUMMARY_PREFERRED_QUERY_STRATEGY):
    if "query_variant_strategy" not in results.columns:
        return None

    model_results = results[results["model"] == model_name]

    if video_id is not None and "video_id" in model_results.columns:
        model_results = model_results[model_results["video_id"] == video_id]

    available = sorted(model_results["query_variant_strategy"].dropna().unique())

    if preferred_strategy in available:
        return preferred_strategy

    if "best_strategy_df" in globals() and not best_strategy_df.empty:
        best_rows = best_strategy_df[best_strategy_df["model"] == model_name]

        if video_id is not None:
            if "video_id" in best_rows.columns:
                best_rows = best_rows[best_rows["video_id"] == video_id]
            else:
                best_rows = pd.DataFrame()

        if not best_rows.empty:
            return best_rows.sort_values("mean_best_iou_at_k", ascending=False)["query_variant_strategy"].iloc[0]

    if "iou" in model_results.columns and not model_results["iou"].dropna().empty:
        return model_results.groupby("query_variant_strategy")["iou"].mean().sort_values(ascending=False).index[0]

    return available[0] if available else None


def refresh_subjectivity_from_prediction_files(results):
    if results.empty:
        return results

    source_paths = []

    if {"source_file", "row_index", "video_id"}.issubset(results.columns):
        source_paths.extend(
            ROOT / str(source_name)
            for source_name in sorted(results["source_file"].dropna().unique())
        )

    if "PREDICTION_SOURCE" in globals():
        source_paths.extend(iter_text_files(PREDICTION_SOURCE))

    source_paths.extend(sorted(PREDICTION_DIR.glob("*_predictions.txt")))

    unique_source_paths = []
    seen_paths = set()

    for source_path in source_paths:
        source_path = Path(source_path)
        resolved = source_path.resolve() if source_path.exists() else source_path

        if resolved in seen_paths:
            continue

        seen_paths.add(resolved)
        unique_source_paths.append(source_path)

    subjectivity_rows = []

    for source_path in unique_source_paths:
        if not source_path.exists():
            continue

        loaded = load_predictions(source_path)

        if loaded.empty:
            continue

        subjectivity_rows.append(
            loaded[["video_id", "source_file", "row_index", "event", "gt_time", "subjectivity"]]
        )

    if not subjectivity_rows:
        return results

    subjectivity_lookup = pd.concat(subjectivity_rows, ignore_index=True)

    exact_keys = ["video_id", "source_file", "row_index"]
    fallback_keys = ["video_id", "event", "gt_time"]

    if set(exact_keys).issubset(results.columns):
        lookup = (
            subjectivity_lookup[exact_keys + ["subjectivity"]]
            .drop_duplicates(exact_keys)
            .rename(columns={"subjectivity": "actual_subjectivity"})
        )
        refreshed = results.merge(lookup, on=exact_keys, how="left")
    elif set(fallback_keys).issubset(results.columns):
        lookup = (
            subjectivity_lookup[fallback_keys + ["subjectivity"]]
            .drop_duplicates(fallback_keys)
            .rename(columns={"subjectivity": "actual_subjectivity"})
        )
        refreshed = results.merge(lookup, on=fallback_keys, how="left")
    else:
        return results

    if "subjectivity" in refreshed.columns:
        refreshed["subjectivity"] = refreshed["actual_subjectivity"].fillna(refreshed["subjectivity"])
    else:
        refreshed["subjectivity"] = refreshed["actual_subjectivity"].fillna(1)

    refreshed["subjectivity"] = refreshed["subjectivity"].astype(int)

    return refreshed.drop(columns=["actual_subjectivity"])


def select_summary_moments(
    results,
    model_name=None,
    query_variant_strategy=None,
    moments_per_video=SUMMARY_MOMENTS_PER_VIDEO,
    padding=SUMMARY_PADDING_SECONDS,
    overlap_threshold=SUMMARY_OVERLAP_THRESHOLD,
):
    if results.empty:
        return pd.DataFrame()

    model_name = model_name or pick_summary_model(results)

    if model_name is None:
        return pd.DataFrame()

    required_cols = {"video_id", "global_rank", "pred_start", "pred_end", "score", "event"}
    missing_cols = sorted(required_cols - set(results.columns))

    if missing_cols:
        raise ValueError(
            "Step 6 expects the raw Step 4 retrieval results_df. "
            f"Missing required column(s): {missing_cols}. Re-run the Step 4 retrieval cell."
        )

    working_results = refresh_subjectivity_from_prediction_files(results.copy())

    if "subjectivity" not in working_results.columns:
        working_results["subjectivity"] = 1

    candidates = working_results[(working_results["model"] == model_name) & (working_results["global_rank"] <= 1)].copy()

    if "subjectivity" in candidates.columns:
        candidates = candidates[candidates["subjectivity"] >= 1]

    if query_variant_strategy is not None and "query_variant_strategy" in candidates.columns:
        candidates = candidates[candidates["query_variant_strategy"] == query_variant_strategy]


    if candidates.empty:
        return pd.DataFrame()

    candidates = candidates.sort_values(
        ["video_id", "subjectivity", "score"],
        ascending=[True, False, False],
    )

    summary_rows = []

    for video_id, group in candidates.groupby("video_id", sort=True):
        selected_strategy = query_variant_strategy or pick_summary_query_strategy(
            working_results,
            model_name,
            video_id=video_id,
        )

        if selected_strategy is not None and "query_variant_strategy" in group.columns:
            group = group[group["query_variant_strategy"] == selected_strategy]

        if group.empty:
            continue

        video_path = VIDEO_DIR / f"video_{int(video_id)}.mp4"
        video_duration = get_video_duration(video_path)
        selected = []
        selected_event_keys = set()

        for item in group.to_dict("records"):
            event_key = normalize_query_text(item["event"]).casefold()

            if event_key in selected_event_keys:
                continue

            summary_start = max(0.0, float(item["pred_start"]) - padding)
            summary_end = min(video_duration, float(item["pred_end"]) + padding)

            if summary_end <= summary_start:
                continue

            overlaps_existing_summary = any(
                temporal_iou(
                    summary_start,
                    summary_end,
                    previous["summary_start"],
                    previous["summary_end"],
                ) >= overlap_threshold
                for previous in selected
            )

            if overlaps_existing_summary:
                continue

            item["summary_start"] = summary_start
            item["summary_end"] = summary_end
            item["summary_time"] = f"{seconds_to_mmss(summary_start)} - {seconds_to_mmss(summary_end)}"
            item["summary_duration"] = summary_end - summary_start
            selected.append(item)
            selected_event_keys.add(event_key)

            if len(selected) >= moments_per_video:
                break

        for summary_order, item in enumerate(sorted(selected, key=lambda row: row["summary_start"]), 1):
            item["summary_order"] = summary_order
            summary_rows.append(item)

    summary = pd.DataFrame(summary_rows)

    if summary.empty:
        return summary

    return summary[
        [
            "model",
            "video_id",
            "query_variant_strategy",
            "summary_order",
            "summary_time",
            "event",
            "score",
            "subjectivity",
            "pred_time",
            "gt_time",
            "iou",
            "summary_start",
            "summary_end",
            "summary_duration",
        ]
    ]


if "results_df" not in globals() or results_df.empty:
    summary_plan_df = pd.DataFrame()
    print("Run Step 4 first; no retrieval results are available for Step 6.")
else:
    summary_model = pick_summary_model(results_df)
    summary_query_strategy = SUMMARY_PREFERRED_QUERY_STRATEGY
    summary_plan_df = select_summary_moments(
        results_df,
        model_name=summary_model,
        query_variant_strategy=summary_query_strategy,
    )
    strategy_label = summary_query_strategy or "best per video"
    print(f"Summary model: {summary_model} | query strategy: {strategy_label}")

    if summary_plan_df.empty:
        print("No summary moments selected.")
    else:
        display(
            summary_plan_df[
                [
                    "video_id",
                    "query_variant_strategy",
                    "summary_order",
                    "summary_time",
                    "event",
                    "score",
                    "subjectivity",
                    "iou",
                ]
            ]
        )

In [ ]:
def format_video_summary(summary_plan, video_id):
    rows = summary_plan[summary_plan["video_id"] == video_id].sort_values("summary_order")

    if rows.empty:
        return f"Video {int(video_id)}: no summary moments selected."

    model_name = rows["model"].iloc[0]
    strategy = rows["query_variant_strategy"].iloc[0] if "query_variant_strategy" in rows.columns else "unknown"
    lines = [f"Video {int(video_id)} retrieval-based summary ({len(rows)} moments, model={model_name}, query_strategy={strategy}):"]

    for row in rows.itertuples(index=False):
        lines.append(f"{int(row.summary_order)}. {row.summary_time}: {row.event}")

    return "\n".join(lines)


summary_text_paths = []

if summary_plan_df.empty:
    print("No summary moments selected.")
else:
    for video_id in sorted(summary_plan_df["video_id"].unique()):
        summary_text = format_video_summary(summary_plan_df, video_id)
        summary_text_path = SUMMARY_OUTPUT_DIR / f"video_{int(video_id)}_retrieval_summary.txt"
        summary_text_path.write_text(summary_text + "\n", encoding="utf-8")
        summary_text_paths.append(
            {
                "video_id": int(video_id),
                "summary_text_path": str(summary_text_path.relative_to(ROOT)),
            }
        )
        print(summary_text)
        print()

    display(pd.DataFrame(summary_text_paths))

In [ ]:
BUILD_SUMMARY_VIDEOS = True
SUMMARY_INCLUDE_AUDIO = False
SUMMARY_VIDEO_CODEC = "libx264"


def _make_subclip(clip, start, end):
    return clip.subclipped(start, end) if hasattr(clip, "subclipped") else clip.subclip(start, end)


def make_summary_video(video_id, summary_plan, output_dir=SUMMARY_OUTPUT_DIR, include_audio=SUMMARY_INCLUDE_AUDIO):
    rows = summary_plan[summary_plan["video_id"] == video_id].sort_values("summary_order")

    if rows.empty:
        return None

    try:
        from moviepy import concatenate_videoclips
    except ImportError:
        from moviepy.editor import concatenate_videoclips

    source_path = VIDEO_DIR / f"video_{int(video_id)}.mp4"
    source_clip = VideoFileClip(str(source_path))
    clips = []
    final_clip = None

    try:
        for row in rows.itertuples(index=False):
            clips.append(_make_subclip(source_clip, float(row.summary_start), float(row.summary_end)))

        final_clip = concatenate_videoclips(clips, method="compose")
        output_dir.mkdir(exist_ok=True)
        output_path = output_dir / f"video_{int(video_id)}_retrieval_summary.mp4"
        final_clip.write_videofile(
            str(output_path),
            codec=SUMMARY_VIDEO_CODEC,
            audio=include_audio,
            logger=None,
        )
        return output_path

    finally:
        if final_clip is not None:
            final_clip.close()

        for clip in clips:
            clip.close()

        source_clip.close()


summary_video_paths = []

if BUILD_SUMMARY_VIDEOS and not summary_plan_df.empty:
    for video_id in sorted(summary_plan_df["video_id"].unique()):
        summary_video_path = make_summary_video(video_id, summary_plan_df)

        if summary_video_path is not None:
            summary_video_paths.append(
                {
                    "video_id": int(video_id),
                    "summary_video_path": str(summary_video_path.relative_to(ROOT)),
                }
            )

if summary_video_paths:
    summary_video_df = pd.DataFrame(summary_video_paths)
    display(summary_video_df)

    for item in summary_video_paths:
        display(Video(str(ROOT / item["summary_video_path"]), embed=True))
else:
    print("No summary videos were rendered. Set BUILD_SUMMARY_VIDEOS = True and run Step 6 after Step 4.")